In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('../scripts')

In [ ]:
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import os
import seaborn as sns
from tqdm import tqdm
import torch
import mintflow

In [19]:
from utils import set_seed
from configs.adata_crc_config import ADATA_ARGS as ADATA_CRC_ARGS
from configs.adata_merfish_config import ADATA_ARGS as ADATA_MERFISH_ARGS
from counterfactual_analysis import compute_lfc_metrics, compute_rmse, compute_edistance, mixing_index


set_seed(0)
DATASET_NAME = "crc"  # Options: "crc", "merfish"

CRC_BASE_PATH = "/data2/a330d/datasets/crc/raw_zenodo"
CRC_SLIDES = ['crc_232' , 'crc_231', 'crc_210', 'crc_221', 'crc_242', 'crc_120']

MERFISH_BASE_PATH = "/data2/a330d/datasets/MERFISH_mouse_brain"
MERFISH_SLIDES = ['C57BL6J-2.036', 'C57BL6J-2.039', 'C57BL6J-2.041']

ADATA_BASE_PATH = CRC_BASE_PATH if DATASET_NAME == "crc" else MERFISH_BASE_PATH
SLIDES = CRC_SLIDES if DATASET_NAME == "crc" else MERFISH_SLIDES
DATA_ARGS = ADATA_CRC_ARGS if DATASET_NAME == "crc" else ADATA_MERFISH_ARGS


LABELS_KEY = DATA_ARGS.get('labels_key')
DOMAINS_KEY = DATA_ARGS.get('domains_key')
N_TOP_GENES = DATA_ARGS.get('n_top_genes')
PATIENT_ID = DATA_ARGS.get('batch_key')
CONTROL_DOMAIN = DATA_ARGS.get('control_domains')[0]  # Assuming only one control domain for simplicity
HOLDOUT_DOMAINS = DATA_ARGS.get('holdout_domains')
N_NEIGHBORS = 5
USE_WANDB = 'False'
MODEL_OUTPUT_PATH = "/data2/a330d/data/ood/trained"
ADATA_SAVE_PATH = f"/data2/a330d/datasets/{DATASET_NAME}/processed"
X_POS = 'CenterX_global_px'
Y_POS = 'CenterY_global_px'

In [21]:
# For crc set domain_key to 'typ' - models were trained before 'typ_clean' was introduced - doesn't change downstream analysis
DOMAINS_KEY = 'typ' if DATASET_NAME == "crc" else DOMAINS_KEY

In [39]:
slide_id = CRC_SLIDES[1]
slide_id

'crc_231'

In [40]:
top_n = 200 # Number of DE genes to use for evaluation
chunk_size = 90000
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
use_recon = False

In [56]:
path_output_files = f"{MODEL_OUTPUT_PATH}/{slide_id}/mintflow/"

In [58]:
# Search os.path.join(path_output_files) directory, and load highest epoch checkpoint
checkpoint_files = [f for f in os.listdir(path_output_files) if f.startswith("checkpoint_epoch_")]
checkpoint_epochs = [int(f.split("checkpoint_epoch_")[1].split(".pt")[0]) for f in checkpoint_files]
latest_epoch = max(checkpoint_epochs)
checkpoint_path = os.path.join(path_output_files, f"checkpoint_epoch_{latest_epoch}.pt")
checkpoint_path

'/data2/a330d/data/ood/trained/crc_231/mintflow/checkpoint_epoch_30.pt'

In [59]:
checkpoint_mintflow = torch.load(
    checkpoint_path,
    map_location='cpu',
    weights_only=False
)
checkpoint_mintflow['model'].to(device)
print("Loaded the checkpoint.")

Loaded the checkpoint.


In [60]:
torch.cuda.empty_cache()

In [61]:
adata = sc.read(
    f"{ADATA_SAVE_PATH}/{slide_id}.h5ad"
)

In [62]:
kwargs_neighbourhood_graph = {
    'spatial_key': 'spatial',
    'library_key': None,
    'set_diag': False,
    'delaunay': False,
    'n_neighs': 5
}
adata.uns = {}

import squidpy as sq
sq.gr.spatial_neighbors(
    adata=adata,
    **kwargs_neighbourhood_graph
)

INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        


In [63]:
num_generated_realisations = 3
n_cells = adata.n_obs
n_genes = adata.n_vars

recon_x = np.zeros((n_cells, n_genes), dtype=np.float32)

graph = adata.obsp["spatial_connectivities"]

indices = np.arange(adata.n_obs)

for start in tqdm(range(0, len(indices), chunk_size), desc="Generating in-silico data in batches"):
    batch_idx = indices[start:start + chunk_size]

    # include neighbors
    neighbor_idx = graph[batch_idx].nonzero()[1]
    all_idx = np.unique(np.concatenate([batch_idx, neighbor_idx]))

    adata_batch = adata[all_idx].copy()

    with torch.no_grad():
        res = mintflow.generate_insilico_ST_data(
            adata=adata_batch,
            obskey_celltype=LABELS_KEY,
            obspkey_neighbourhood_graph="spatial_connectivities",
            device=device,
            batch_index_trainingdata=0,
            num_generated_realisations=num_generated_realisations,
            model=checkpoint_mintflow["model"],
            data_mintflow=checkpoint_mintflow["data_mintflow"],
            dict_all4_configs=checkpoint_mintflow["dict_all4_configs"],
            estimate_spatial_sizefactors_on_sections=[0]
        )

    # compute mean reconstruction across realizations
    recon_batch = np.stack([
        r["MintFlow_Generated_Xint"] + r["MintFLow_Generated_Xmic"]
        for r in res["list_generated_realisations_ie_expressions"]
    ]).mean(0)

    # map results back to global indices
    pos = np.searchsorted(all_idx, batch_idx)

    recon_x[batch_idx] = recon_batch[pos]

    torch.cuda.empty_cache()

Generating in-silico data in batches:   0%|          | 0/3 [00:00<?, ?it/s]

Evaluating on tissue section: 0:   0%|          | 0/603 [00:00<?, ?it/s]

/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "


Generating the realisations of the expression data (i.e. generative samples) for the provided in silico tissue…

/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/mintflow/generativemodel.py:546: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(edge_index),
/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
Generating in-silico data in batches:  33%|███▎      | 1/3 [02:41<05:22, 161.11s/it]

...

/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/mintflow/generativemodel.py:546: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(edge_index),
/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
Generating in-silico data in batches:  67%|██████▋   | 2/3 [05:23<02:41, 161.97s/it]

Evaluating on tissue section: 0:   0%|          | 0/603 [00:00<?, ?it/s]

/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "


Generating the realisations of the expression data (i.e. generative samples) for the provided in silico tissue…

/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/mintflow/generativemodel.py:546: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(edge_index),
/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
Generating in-silico data in batches: 100%|██████████| 3/3 [07:50<00:00, 156.89s/it]


In [66]:
adata.obsm['recon_x'] = recon_x

# Predict for all cell types

In [ ]:
target_celltypes = ["Fibroblast", "Endothelial", "Myeloid", "T_cell", "Epithelial"]
results = []

for target_ct in tqdm(target_celltypes, desc="Target Cell Types"):
    is_holdout_ct = adata.obs[LABELS_KEY] == target_ct
    is_in_control_domain = adata.obs[DOMAINS_KEY].str.contains(CONTROL_DOMAIN)
    control_mask = is_holdout_ct & is_in_control_domain
    control_idx = np.where(control_mask)[0]
    control = adata.layers['counts'][control_mask.values, :].todense()
    control = np.asarray(control)
    if use_recon:
        control = adata.obsm['recon_x'][control_mask.values, :]

    for hd in HOLDOUT_DOMAINS:
        is_in_holdout_domain = adata.obs[DOMAINS_KEY].str.contains(hd)
        target_mask = is_holdout_ct & is_in_holdout_domain
        target_idx = np.where(target_mask)[0]
        perturbed_x = np.zeros((adata.n_obs, adata.n_vars), dtype=np.float32)

        for start in tqdm(range(0, len(target_idx), chunk_size), desc="Generating in-silico data in batches"):
            batch_cells = target_idx[start:start + chunk_size]

            # include neighbors
            neighbor_idx = graph[batch_cells].nonzero()[1]
            all_idx = np.unique(np.concatenate([batch_cells, neighbor_idx]))

            adata_batch = adata[all_idx].copy()
            adata_batch.uns = {}

            # map global indices → batch indices
            batch_pos = np.searchsorted(all_idx, batch_cells)

            # choose control replacements
            sampled_control = np.random.choice(control_idx, size=len(batch_cells), replace=True)

            # replace gene expression for target cells
            adata_batch.X[batch_pos] = adata.X[sampled_control]

            with torch.no_grad():
                res = mintflow.generate_insilico_ST_data(
                    adata=adata_batch,
                    obskey_celltype=LABELS_KEY,
                    obspkey_neighbourhood_graph="spatial_connectivities",
                    device=device,
                    batch_index_trainingdata=0,
                    num_generated_realisations=num_generated_realisations,
                    model=checkpoint_mintflow["model"],
                    data_mintflow=checkpoint_mintflow["data_mintflow"],
                    dict_all4_configs=checkpoint_mintflow["dict_all4_configs"],
                    estimate_spatial_sizefactors_on_sections=[0]
                )

            recon_batch = np.stack([
                r["MintFlow_Generated_Xint"] + r["MintFLow_Generated_Xmic"]
                for r in res["list_generated_realisations_ie_expressions"]
            ]).mean(0)

            perturbed_x[batch_cells] = recon_batch[batch_pos]

            torch.cuda.empty_cache()

        # Compute metrics
        counterfactual = perturbed_x[target_mask]
        target = adata.layers['counts'][target_mask.values, :].todense()
        target = np.asarray(target)
        if use_recon:
            target = adata.obsm['recon_x'][target_mask.values, :]

        pear, spear, prec, dir_match, deg = compute_lfc_metrics(control=control, target=target, counterfactual=counterfactual, n_deg=top_n)
        rmse = compute_rmse(observed=target, predicted=counterfactual, deg=deg)
        edist_global = compute_edistance(observed=target, predicted=counterfactual, deg=deg)
        edist_local = compute_edistance(observed=target, predicted=counterfactual, deg=deg, local=True)
        mix_idx = mixing_index(observed=target, predicted=counterfactual)
        _, _, _, dir_match_k, _ = compute_lfc_metrics(control=control, target=target, counterfactual=counterfactual, n_deg=top_n, direction_match_normalize="k")

        results.append(
            dict(
                dataset_name=DATASET_NAME,
                sid=slide_id,
                control_domain=CONTROL_DOMAIN,
                target_domain=hd,
                n_deg=top_n,
                model_name="mintflow",
                holdout_celltype=target_ct,
                spearman=spear,
                pearson=pear,
                precision=prec,
                direction_match=dir_match,
                direction_match_k=dir_match_k,
                mixing_index=mix_idx,
                edistance_global=edist_global,
                edistance_local=edist_local,
                rmse=rmse,
            )
        )

Target Cell Types:   0%|          | 0/5 [00:00<?, ?it/s]/data/a330d/miniforge3/envs/mintflow_env/lib/python3.11/site-packages/scipy/sparse/_index.py:201: SparseEfficiencyWarning: Changing the sparsity structure of a csc_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray_sparse(i, j, x)


Evaluating on tissue section: 0:   0%|          | 0/603 [00:00<?, ?it/s]

In [38]:
df_results = pd.DataFrame(results)
df_csv_fname = f"mintflow_{slide_id}.csv"
os.makedirs(os.path.join('../results/mintflow_crc'), exist_ok=True)
df_results.to_csv(os.path.join('../results/mintflow_crc', df_csv_fname), index=False)